# 11. Как понять, что модель учится

**Цель:** научиться читать train/validation loss и делать выводы, которые действительно поддерживают измерения.

Это не урок новой архитектуры и не попытка любой ценой получить красивый текст. Ты уже написал обучение и генерацию; теперь добавишь к ним осмысленное наблюдение.

**Ты пишешь:** оценку среднего loss без обучения и сбор метрик через каждые 50 шагов. **Ментор даёт:** прежнюю модель, твои функции из урока 10, фиксированные батчи и проверки для самостоятельного запуска.

Начни с **A**, без запуска обучения. Части B–D можно делать на следующем занятии. Не нужно завершать весь ноутбук за один вечер.

Подробная теория: [Как читать loss](../../notes/interview-prep/08n-reading-training-results.md). Ноутбук независим от запущенного kernel урока 10: нужные определения скопированы сюда, старый ноутбук не исполняется. Используется локальный корпус урока 09; интернет не нужен. Ячейки с NotImplementedError намеренно оставлены тебе. Ментор не запускал этот урок.

## A. Сначала научимся читать числа

Training loss отвечает на вопрос о предсказании на обучающих данных. Validation loss — на данных, по которым мы не обновляли веса. В этом уроке обе оценки считаем одинаковым способом, в режиме eval, на фиксированных наборах окон.

Loss — не процент ошибок и не оценка литературного качества. Меньше означает, что модель в среднем назначает правильным следующим токенам более высокие вероятности в смысле log-loss.

Ниже **вымышленные учебные примеры**, а не результаты нашей модели. Четырёх строк достаточно для первого разбора, поэтому используем таблицы, не добавляя код графиков.

| Шаг | Случай I: train | I: val | Случай II: train | II: val | Случай III: train | III: val |
|---:|---:|---:|---:|---:|---:|---:|
| 0 | 4.5 | 4.5 | 4.5 | 4.5 | 4.5 | 4.5 |
| 100 | 3.0 | 3.1 | 2.7 | 3.0 | 4.4 | 4.5 |
| 200 | 2.6 | 2.7 | 1.8 | 3.2 | 4.5 | 4.4 |
| 300 | 2.4 | 2.5 | 1.1 | 3.6 | 4.4 | 4.5 |

Не ставь диагноз по одному числу. Ищи направление изменения обеих метрик. Разрыв train/val сам по себе ещё не доказывает переобучение; важна динамика и надёжность измерения.

### Мои ответы — A

1. В каком случае улучшение есть и на train, и на validation?
2. Где видны признаки переобучения? Какие именно числа это показывают?
3. Где убедительного улучшения почти нет? Назови две возможные причины, но обозначь их как гипотезы.
4. В случае II какую из четырёх точек ты выбрал бы по validation loss? Почему не обязательно последнюю?

**Остановка:** пришли ответы A. До нашего разбора обучение запускать не нужно.

Ответы:
1. Если хорошее, стабильное обучние
2. Если на обучающей выборке ошибка уменьшается, а на валидационной растет
3. Если train и  validation не меняются или скачут на протяжении нескольких шагов
4. вторую, т.к. у нее меньше ошибка

### Разбор ментора

Исходные ответы сохранены. Разбор относится к вымышленной таблице части A, а не к результатам реального обучения. Код не запускался.

1. Ты описал общую идею, но вопрос просил выбрать случай. Это случай I: train снижается 4.5 → 3.0 → 2.6 → 2.4, validation — 4.5 → 3.1 → 2.7 → 2.5. Наблюдаем улучшение на обоих наборах. Это не доказывает отсутствие переобучения в будущем и не означает, что текст уже связный.

2. Признак назван правильно. В таблице это случай II после шага 100: train продолжает снижаться 2.7 → 1.8 → 1.1, а validation растёт 3.0 → 3.2 → 3.6. Именно расхождение направлений после минимума validation поддерживает вывод о признаках переобучения; не просто наличие разницы между двумя loss.

3. Твоё описание соответствует случаю III: обе оценки остаются около 4.4–4.5. Не хватает двух гипотез. Например: (а) learning rate слишком мал, поэтому за эти 300 шагов веса меняются недостаточно; (б) ошибка в цикле обучения — не вызывается optimizer.step(), поэтому параметры не обновляются. Это возможные объяснения, не установленный диагноз. Для (а) можно сравнить контролируемый запуск с другим learning rate после проверки кода; для (б) проверить вызовы backward/step, градиенты и изменение весов. Сами небольшие колебания таблицы не позволяют установить причину.

4. Верно: вторая точка — шаг 100, validation loss = 3.0, минимум среди четырёх измерений. На последнем шаге train лучше, но validation хуже (3.6). Мы выбираем по validation, а не по минимальному train loss. Это лучшая из измеренных точек, а не доказанный минимум между ними. Чтобы использовать её модель после дальнейшего обучения, нужно заранее сохранить её веса: одна таблица метрик их не восстанавливает.

Итог: принцип переобучения и выбор точки ты понимаешь. Закрепляем форму ответа: номер случая → конкретные числа → осторожный вывод. Часть A разобрана; можно переходить к B и самостоятельно написать evaluate_batches.

## Подготовка для практики — готовый код

Дальше пригодится kernel deepschool с PyTorch. Используем тот же корпус: train I–X, validation XI–XII, test XIII–XIV не оцениваем. Источник и ограничения извлечения — `data/captains_daughter/README.md`.

Код модели и твои основные функции перенесены из урока 10. Это не новая работа, которую нужно повторять по памяти. Обучение остаётся на 300 шагах; размер модели и learning rate не меняем.

In [1]:
import json
import math
import unicodedata
from pathlib import Path

import torch
from torch import nn
from torch.nn import functional as F

SEED = 42
CONTEXT_LENGTH = 32
BATCH_SIZE = 16
TRAIN_STEPS = 300
LEARNING_RATE = 1e-3
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(SEED)

candidates = [p for base in [Path.cwd(), *Path.cwd().parents]
              for p in [base, base / 'Seminars' / 'transformers']]
LESSON_DIR = next((p for p in candidates if (p / 'data/captains_daughter/chapters.json').is_file()), None)
if LESSON_DIR is None:
    raise FileNotFoundError('Открой ноутбук внутри проекта deepschool с локальным корпусом урока 09')
chapters = json.loads((LESSON_DIR / 'data/captains_daughter/chapters.json').read_text(encoding='utf-8'))
def normalize(text):
    return unicodedata.normalize('NFC', text).lower()
train_text = normalize('\n\n'.join(c['text'] for c in chapters[:10]))
val_text = normalize('\n\n'.join(c['text'] for c in chapters[10:12]))
tokens = ['<unk>'] + sorted(set(train_text))
stoi = {token: i for i, token in enumerate(tokens)}
vocab_size = len(tokens)
def encode(text):
    return torch.tensor([stoi.get(c, 0) for c in normalize(text)], dtype=torch.long)
def decode(ids):
    return ''.join(tokens[i] for i in ids.tolist())
train_ids, val_ids = encode(train_text), encode(val_text)
print('device:', DEVICE, '| V:', vocab_size)
print('train / val characters:', len(train_ids), len(val_ids))

device: cuda | V: 75
train / val characters: 130505 29122


In [2]:
def make_example(data, start, context_length):
    x = data[start:start+context_length]
    y = data[start+1:start+context_length+1]
    return x,y

In [3]:
def make_batch(data, starts, context_length):

    xs, ys = list(), list()
    for start in starts:
        x, y = make_example(data, start, context_length)
        xs.append(x)
        ys.append(y)
    xb = torch.stack(xs, dim=0).to(DEVICE)
    yb = torch.stack(ys, dim=0).to(DEVICE)

    return xb, yb

In [4]:
def sample_batch(data, generator):
    # high исключается: последний допустимый start = len(data) - T - 1.
    starts = torch.randint(len(data) - CONTEXT_LENGTH, (BATCH_SIZE,), generator=generator)
    xb, yb = make_batch(data, starts.tolist(), CONTEXT_LENGTH)
    return xb.to(DEVICE), yb.to(DEVICE)

In [5]:
class VectorizedCausalAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        if d_model <= 0 or num_heads <= 0 or d_model % num_heads != 0:
            raise ValueError('Нужны положительные D,H и D, кратное H')
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.query_projection = nn.Linear(d_model, d_model, bias=False)
        self.key_projection = nn.Linear(d_model, d_model, bias=False)
        self.value_projection = nn.Linear(d_model, d_model, bias=False)
        self.output_projection = nn.Linear(d_model, d_model, bias=False)

    def project_heads(self, x):
        if x.ndim != 3 or x.shape[-1] != self.d_model or x.shape[1] == 0:
            raise ValueError('Ожидается непустой x формы (B,T,D)')
        batch_size, length, _ = x.shape
        # Готовая реализация из урока 05c: проекция и разделение голов.
        q, k, v = [
            projection(x)
            .reshape(batch_size, length, self.num_heads, self.head_dim)
            .transpose(1, 2)
            for projection in (
                self.query_projection,
                self.key_projection,
                self.value_projection,
            )
        ]

        return q, k, v

    def forward(self, x):
        batch_size, length, _ = x.shape
        q, k, v = self.project_heads(x)
        scores = q @ k.transpose(-2, -1) / math.sqrt(q.shape[-1])
        positions = torch.arange(x.shape[1], device=x.device)
        blocked = positions[None, :] > positions[:, None]
        masked_scores = scores.masked_fill(blocked, float('-inf'))
        weights = torch.softmax(masked_scores, dim=-1)
        context = weights @ v
        combined = context.transpose(1, 2).reshape(batch_size, length, self.d_model)
        return self.output_projection(combined)

In [6]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.norm_1 = nn.LayerNorm(d_model)
        self.attention = VectorizedCausalAttention(d_model, num_heads)
        self.norm_2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x):
        x = x + self.attention(self.norm_1(x))
        x = x + self.mlp(self.norm_2(x))
        return x

In [7]:
class TinyLanguageModel(nn.Module):
    def __init__(self, vocab_size, max_seq_len=16, d_model=8,
                 num_heads=2, d_ff=32, n_layers=1):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_seq_len, d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff)
            for _ in range(n_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, token_ids):
        if token_ids.ndim != 2:
            raise ValueError('Ожидаются token IDs формы (B,T)')
        length = token_ids.shape[1]
        if not 1 <= length <= self.max_seq_len:
            raise ValueError('Длина должна быть от 1 до max_seq_len')
        positions = torch.arange(length, device=token_ids.device)
        x = self.token_embedding(token_ids) + self.position_embedding(positions)
        for block in self.blocks:
            x = block(x)
        x = self.final_norm(x)
        logits = self.lm_head(x)
        return logits

In [8]:
def new_model():
    return TinyLanguageModel(vocab_size=vocab_size, max_seq_len=CONTEXT_LENGTH,
                             d_model=32, num_heads=4, d_ff=128, n_layers=2).to(DEVICE)

In [9]:
def language_model_loss(logits, targets):
    V = logits.shape[-1]
    logits = logits.reshape(-1, V)
    targets = targets.reshape(-1)
    loss = F.cross_entropy(logits, targets)
    return loss

In [10]:
def train_step(model, optimizer, xb, yb):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    logits = model(xb)

    loss = language_model_loss(logits, yb)
    loss.backward()
    optimizer.step()
    return loss.item()

In [11]:
@torch.no_grad()
def generate_ids(model, idx, max_new_tokens, context_length):
    model.eval()
    for _ in range(max_new_tokens):
        logits = model(idx[:,-context_length:])  # B, T, V
        last_logits = logits[:, -1, :]             # B,V
        probs = torch.softmax(last_logits, dim=-1)  # B,V
        next_token = torch.multinomial(probs, num_samples=1) #  B,1
        idx = torch.cat([idx, next_token], dim=1)
    # Напиши сам: окно -> logits -> последний logits -> probabilities
    # -> один ID для каждой строки -> дополнение полной истории.
    # Верни idx после цикла. Здесь не кодируем и не декодируем текст.
    return idx

## B. Оценка без обучения

Раньше evaluate была готовой. Теперь напишешь её сам, но выбор окон уже подготовлен.

**Зачем фиксированные батчи?** Если каждый раз брать новые окна, loss меняется и из-за весов, и из-за трудности новых примеров. Те же окна помогают сравнить этапы обучения. Это небольшая выборка, а не весь корпус; результат всё ещё имеет ограничения.

Для каждой оценки используем 4 батча по 16 последовательностей длиной 32. У всех одинаковое число targets, поэтому можно усреднить четыре batch loss обычным средним. При разных размерах нужно взвешивать по числу учитываемых targets. Здесь нет padding и исключённых позиций.

Вход evaluate_batches: модель и список пар `(xb,yb)`. Выход: одно Python-число среднего loss.

Порядок работы:

1. Сохрани `model.training`, чтобы затем вернуть прежний режим.
2. Включи eval; no_grad уже задан декоратором.
3. Для каждого батча получи logits, вызови свою language_model_loss и запомни loss.item().
4. Верни среднее. Восстановление режима в finally уже написано.

**Здесь нет zero_grad, backward или step:** мы только измеряем. Старые .grad могут оставаться после обучения — их наличие не означает, что evaluate создала новые градиенты. eval и no_grad решают разные задачи.

In [12]:
# Готовые независимые наборы для измерения. Не используем train_rng.
def fixed_batches(data, seed, count=4):
    rng = torch.Generator().manual_seed(seed)
    return [sample_batch(data, rng) for _ in range(count)]

train_eval_batches = fixed_batches(train_ids, 1001)
val_eval_batches = fixed_batches(val_ids, 1002)

@torch.no_grad()
def evaluate_batches(model, batches):
    if not batches:
        raise ValueError('Нужен хотя бы один батч')
    was_training = model.training
    model.eval()
    try:
        values = []

        for xb, yb in batches:
            logits = model(xb)     # Предсказания модели
            loss = language_model_loss(logits, yb)         # Твоя функция вычисления loss
            values.append(loss.item()) # Python-число

        return sum(values) / len(values)       # Среднее списка values
    finally:
        model.train(was_training)

In [13]:
# Проверка на отдельной модели; запускаешь после своего решения B.
torch.manual_seed(SEED)
probe = new_model()
probe.train()
before = {name: p.detach().clone() for name, p in probe.named_parameters()}
first = evaluate_batches(probe, val_eval_batches)
second = evaluate_batches(probe, val_eval_batches)
assert isinstance(first, float) and math.isfinite(first)
assert math.isclose(first, second, rel_tol=1e-6, abs_tol=1e-6)
assert probe.training, 'Нужно восстановить прежний режим'
assert all(torch.equal(before[name], p.detach()) for name, p in probe.named_parameters())
assert all(p.grad is None for p in probe.parameters())
print('B: оценки совпали, веса не менялись, режим восстановлен:', first)
del probe, before

B: оценки совпали, веса не менялись, режим восстановлен: 4.5059205293655396


### Мои ответы — B

1. Почему при оценке не вызываем optimizer.step()?
2. Для чего здесь и eval, и no_grad?
3. Почему нельзя сравнить loss случайного трудного батча на шаге 50 с лёгким батчем на шаге 100 и уверенно приписать разницу обучению?

Ответы:
1. Потому что при оцнеке не меняем веса
2. чтобы не обновлялись веса и не считались градиенты
3. меньший loss может объясняться более лёгкими примерами, а не улучшением модели.

### Разбор ментора

Исходные ответы сохранены. Проверено чтением кода и сохранённых outputs, без повторного запуска.

Код B верен: перебираешь переданный batches, считаешь средний loss и восстанавливаешь режим в finally. Сохранённая проверка сообщает, что повторные оценки совпали, веса не изменились и режим восстановлен.

Ответы 1 и 3 верны. Ответ 2 нужно разделить: model.eval() переключает поведение модулей вроде dropout, но не запрещает вычисление градиентов или изменение весов. @torch.no_grad() отключает запись графа autograd при оценке. Веса здесь остаются прежними потому, что мы не выполняем обновление — нет optimizer.step() и других операций изменения параметров. В нашей модели нет dropout/BatchNorm, однако явное разделение режима и autograd остаётся правильным.

## C. Измеряем динамику, а не только начало и конец

Теперь используешь знакомые sample_batch и train_step, а через каждые 50 обновлений вызываешь свою evaluate_batches для train и val.

В журнале одна запись — словарь с ключами `step`, `train_loss`, `val_loss`. На старте есть точка 0. Далее нужны 50, 100, 150, 200, 250, 300: всего семь записей.

Важно: train_loss в этом журнале — оценка на фиксированных train-окнах **после** указанного числа обновлений. Это не loss последнего обучающего батча. Так train и validation измеряются сопоставимо.

Перед повторением эксперимента запускай ячейку инициализации ниже: иначе продолжишь обучение старой модели. Новую модель или Adam внутри цикла не создаём.

In [14]:
# Сброс эксперимента. Здесь нет скрытой загрузки старых весов.
torch.manual_seed(SEED)
model = new_model()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
train_rng = torch.Generator().manual_seed(43)
EVAL_INTERVAL = 50
metrics = [{'step': 0,
            'train_loss': evaluate_batches(model, train_eval_batches),
            'val_loss': evaluate_batches(model, val_eval_batches)}]
print(metrics[0])

{'step': 0, 'train_loss': 4.501575827598572, 'val_loss': 4.5059205293655396}


In [15]:
for step_id in range(1, TRAIN_STEPS+1):
    xb, yb = sample_batch(train_ids, train_rng)
    loss = train_step(model, optimizer, xb, yb)

    if step_id % EVAL_INTERVAL == 0:
        train_eval_loss = evaluate_batches(model, train_eval_batches)
        val_eval_loss = evaluate_batches(model, val_eval_batches)

        record = {
            'step': step_id,
            'train_loss': train_eval_loss,
            'val_loss': val_eval_loss,
        }

        metrics.append(record)
        print(record)

{'step': 50, 'train_loss': 3.2271618843078613, 'val_loss': 3.226428508758545}
{'step': 100, 'train_loss': 2.9784446954727173, 'val_loss': 2.9686185717582703}
{'step': 150, 'train_loss': 2.823496162891388, 'val_loss': 2.8111937046051025}
{'step': 200, 'train_loss': 2.740165591239929, 'val_loss': 2.7179237008094788}
{'step': 250, 'train_loss': 2.6949050426483154, 'val_loss': 2.674671232700348}
{'step': 300, 'train_loss': 2.656249701976776, 'val_loss': 2.635077178478241}


In [16]:
assert [r['step'] for r in metrics] == list(range(0, TRAIN_STEPS + 1, EVAL_INTERVAL))
assert all(math.isfinite(r[key]) for r in metrics for key in ['train_loss', 'val_loss'])
print(f"{'step':>5} {'train loss':>12} {'val loss':>12}")
for row in metrics:
    print(f"{row['step']:5d} {row['train_loss']:12.4f} {row['val_loss']:12.4f}")
# Здесь намеренно нет assert на убывание каждого loss.

 step   train loss     val loss
    0       4.5016       4.5059
   50       3.2272       3.2264
  100       2.9784       2.9686
  150       2.8235       2.8112
  200       2.7402       2.7179
  250       2.6949       2.6747
  300       2.6562       2.6351


### Мои ответы — C

1. Запиши реальные начальные и конечные метрики из своей таблицы.
2. Какие признаки улучшения или ухудшения видишь? Укажи конкретные шаги и числа.
3. Видны ли признаки переобучения? Отдели наблюдение от предположения; ответ «пока недостаточно данных» допустим.
4. На каком измеренном шаге validation loss минимален? Можно ли по этой таблице восстановить веса того шага?

Подсказка к пункту 4: мы сохранили числа, но не checkpoint. Выбор лучшего шага и сохранение его весов — разные действия.

Ответы:
1. Начальные 4.5016       4.5059, конечные 2.6562       2.6351
2. у нас обучение на всех шагах, уменьшился лосс до 2.6351
3. нет, нет признаков
4. на последнем. Нет, веса нельзя восстановить

### Разбор ментора

Исходные ответы сохранены.

Код C верен: каждый шаг выбирает обучающий батч и обновляет модель; каждые 50 шагов отдельно измеряются два loss на фиксированных наборах. Новая запись добавляется через append. В сохранённой таблице семь точек: 0, 50, 100, 150, 200, 250, 300.

1. Метрики записаны правильно: train 4.5016 → 2.6562, validation 4.5059 → 2.6351.
2. Уточнение формулировки: обе оценки снижаются во всех ИЗМЕРЕННЫХ точках. Мы не измеряли их после каждого из 300 обновлений и не утверждаем, что каждый отдельный шаг улучшал результат. Например, с шага 100 до 150 train снизился 2.9784 → 2.8235, validation — 2.9686 → 2.8112. Значение 2.6351 — именно конечный validation loss.
3. Верно: в этих измерениях нет явных признаков переобучения — validation не начинает расти при снижении train. Это ограниченный вывод о данном запуске и выбранных окнах, не доказательство полного отсутствия переобучения.
4. Верно: минимум среди измеренных точек на шаге 300, validation = 2.6351. Таблица не позволяет восстановить параметры. Нюанс: пока kernel не перезапущен, в переменной model уже находятся последние веса шага 300; это не восстановление по метрикам. Промежуточные веса без checkpoint не сохранены.

## D. Почему числа и текст — не одно и то же

При validation на вход подаём настоящий фрагмент книги и проверяем следующий символ на каждой позиции. Такой режим часто называют teacher forcing: предыдущие токены берём из данных, а не из предсказаний модели.

При генерации модель сама выбирает продолжение. Неудачный символ становится частью следующего входа. Поэтому хороший средний результат на настоящих контекстах ещё не гарантирует связную длинную генерацию.

Улучшение частот букв, пробелов и коротких сочетаний уже может уменьшать loss. Из этого нельзя заключить, что модель научилась удерживать смысл и сюжет.

Ниже используем твою генерацию без temperature. Берём два seed заранее и сохраняем оба результата, не выбираем только красивый. Это качественное наблюдение, не надёжная численная метрика качества текста. Модель остаётся на последнем шаге, не обязательно на лучшем.

In [17]:
prompt = encode('отец мой ').unsqueeze(0).to(DEVICE)
for seed in [2026, 2027]:
    torch.manual_seed(seed)
    generated = generate_ids(model, prompt.clone(), 100, CONTEXT_LENGTH)
    print('seed:', seed)
    print(decode(generated[0].cpu()))
    print()

seed: 2026
отец мой н вроя м оствилорету сис, я валь восвово ч:дчндду, забе у.асл карыелавоврчелристро!жрра крянгол гос 

seed: 2027
отец мой витатя. с ссмнема педанозадитьй бьго, нсведоебнре иричия мо. мего зо— дри си воало воилстутл пошл  н



### Мои ответы — D

1. Что наблюдаешь в двух продолжениях: пробелы, похожие на слова фрагменты, целые слова, связные предложения? Приведи короткие примеры, не преувеличивая качество.
2. Почему разные продолжения одной модели не означают, что веса изменились?
3. Напиши итог из трёх предложений: измеренный результат → ограничение измерения → следующий эксперимент.

Не требуем осмысленного текста после 300 шагов. Если он бессмысленный, это наблюдение для отчёта, а не причина переписывать весь код.

Ответы:
1. Там пока бессмысленный текст
2. Потому что у нас следующий токен берется не одинаковый, а через multinomial
3.Результат: «За 300 шагов train loss снизился с Начальные 4.5016 4.5059, конечные 2.6562 2.6351.
Ограничение: «Это показывает что модель обучается но не доказывает что может генерировать слова.
Следующий эксперимент: «При неизменных весах я сравню генерацию с разной temperature, чтобы понять, что будет, если модель будет выбирать более случайно новый токен» Мне кажется, что станет еще хуже, она и сейчас то слова не может составить.

### Разбор ментора

Заполним после твоей попытки. Исходные ответы сохраним.

## Завершение и следующий шаг

Урок готов к разбору, когда ты можешь объяснить, что именно измеряет loss, почему оценка не обучает модель и какие выводы поддерживает твоя таблица. Ты не обязан получить заранее заданные числа или увидеть переобучение.

Test-главы не трогаем. Фиксированные validation-окна помогают сравнению, но многократный подбор решений под них тоже может переобучить наш выбор; они не заменяют независимую итоговую оценку.

Следующий урок: temperature и управляемый эксперимент с генерацией без изменения весов. Сегодня новых параметров генерации не добавляем.

Проверка ментора будет по твоему коду, ответам и сохранённым outputs. Перед отправкой сохрани ноутбук. Коммит — после разбора и по отдельному решению.